# __MODEL_LABEL_MARKDOWN__: data ingestion

Load and clean the source data, record its provenance, and save the dataset
for model training. Declare model transforms in notebook 03.


In [ ]:
DATABASE_MODE = __DATABASE_MODE_LITERAL__  # "local" or "remote"
RUNTIME_MODULE = __RUNTIME_MODULE_LITERAL__  # e.g. "project_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = __EXPECTED_REMOTE_DATABASE_LITERAL__
ALLOW_REMOTE_WRITES = False

DATA_AS_OF = ""  # Required ISO date: the dataset version, not a deployment date.
REPLACE_DATASET = False


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

import numpy as np
import pandas as pd

from pricing_pipeline.notebook import PricingDataset, connect

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"
DATASET_PATH = MODEL_DIR / ".local" / "dataset.joblib"


## Connect


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)


## Read source data

Replace this synthetic example with your SQL query or file read. Keep the
source keys, target, source feature columns, and data-as-of date.


In [ ]:
if not DATA_AS_OF.strip():
    raise ValueError("Set the required DATA_AS_OF dataset version stamp.")
rng = np.random.default_rng(42)
raw_df = pd.DataFrame({
    "__PRIMARY_KEY__": np.arange(1, 101),
    "__FEATURE_NAME__": rng.normal(size=100),
    "segment": rng.choice(["A", "B", "C"], size=100),
    "data_as_of": [DATA_AS_OF] * 100,
})
raw_df["__TARGET_NAME__"] = rng.poisson(
    np.exp(
        -0.5
        + 0.25 * raw_df["__FEATURE_NAME__"]
        + raw_df["segment"].map({"A": 0.0, "B": 0.2, "C": -0.1})
    )
)
display({"Rows loaded": len(raw_df), "Columns loaded": len(raw_df.columns)})


## Prepare the dataset

Clean and order the source rows here. Keep the source columns needed by
model transforms in notebook 03.


In [ ]:
df = (
    raw_df.loc[
        :,
        [
            "__PRIMARY_KEY__",
            "__TARGET_NAME__",
            "__FEATURE_NAME__",
            "segment",
            "data_as_of",
        ],
    ]
    .sort_values("__PRIMARY_KEY__")
    .reset_index(drop=True)
)
display(df.head())


## Record provenance and save

The handoff contains the source data and its provenance. Saving verifies its
contents and protects an existing artifact unless replacement is enabled.


In [ ]:
dataset = PricingDataset(
    df=df,
    name="__DATASET_NAME__",
    source="replace_with_source_name",
    key="__PRIMARY_KEY__",
    as_of="data_as_of",
)
dataset.save(DATASET_PATH, replace=REPLACE_DATASET)
display(dataset)
